In [229]:
import pandas as pd
import numpy as np
import scipy as scp
import dgl 
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
ROOT_PATH = '/Users/keltonn/Desktop/Repo/CS5228-project'

# Train Stations

In [9]:
mrt_node = pd.read_csv(f'{ROOT_PATH}/00_location_graph_data_mining/raw_graph_data/sg-mrt-nodes.csv')
mrt_node

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False
...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False


In [11]:
mall_feature = pd.read_csv(f'{ROOT_PATH}/data/collected_data/mrt_data/sg-station-mall-counts.csv')
mall_feature.columns = ['NAME','n_malls_500m','nearest_mall','nearest_mall_m','malls_500m']
mall_feature = mall_feature[['NAME','n_malls_500m','malls_500m','nearest_mall','nearest_mall_m',]]
mall_feature

,NAME,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...
187,woodleigh mrt station,0,NaN,NEX,1300.3
188,xilin mrt station,0,NaN,Changi City Point,617.4
189,yew tee mrt station,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,0,NaN,AMK Hub,1450.1


In [14]:
df = pd.merge(mrt_node, mall_feature, how = 'left', on = 'NAME')
df

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False,0,NaN,NEX,1300.3
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False,0,NaN,Changi City Point,617.4
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False,0,NaN,AMK Hub,1450.1


In [15]:
df.to_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-keys.csv', index = False)

# Train Traffic

In [16]:
dates = ['202605','202606','202607']
traffic_lst = []
for d in dates:
    traffic = pd.read_csv(f'{ROOT_PATH}/data/collected_data/mrt_data/sg-mrt-nodes-features-{d}.csv')
    traffic = traffic[traffic['pv_matched'] == True].reset_index(drop = True)
    traffic = traffic[['NAME',
        'pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
        'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
        'pv_busiest_hour', 'year_month',]]
    traffic_lst.append(traffic)

traffic_df = pd.concat(traffic_lst).reset_index(drop = True)
traffic_df


,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour,year_month
0,admiralty mrt station,1220591.0,1088699.0,2309290.0,1601448.0,707842.0,447168.0,423421.0,7.0,2026-05
1,aljunied mrt station,701918.0,708113.0,1410031.0,874144.0,535887.0,210393.0,239064.0,18.0,2026-05
2,ang mo kio mrt station,1220599.0,1339014.0,2559613.0,1726662.0,832951.0,408605.0,466877.0,18.0,2026-05
3,bartley mrt station,254624.0,228521.0,483145.0,353537.0,129608.0,101289.0,86506.0,7.0,2026-05
4,bayfront mrt station,1024518.0,966289.0,1990807.0,1046697.0,944110.0,133522.0,254453.0,20.0,2026-05
...,...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,402335.0,373959.0,776294.0,637179.0,139115.0,176196.0,164702.0,7.0,2026-07
428,woodleigh mrt station,467808.0,480973.0,948781.0,741200.0,207581.0,167851.0,211816.0,18.0,2026-07
429,yew tee mrt station,919637.0,886547.0,1806184.0,1434612.0,371572.0,376052.0,383765.0,7.0,2026-07
430,yio chu kang mrt station,687769.0,640383.0,1328152.0,1157293.0,170859.0,346223.0,308066.0,8.0,2026-07


In [17]:
traffic_df.to_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-traffic.csv', index= False)

# Traffic Aggregation

In [195]:
num_col = ['pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
       'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
       ]
cat_col = ['pv_busiest_hour',]


traffic_df = pd.read_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-traffic.csv')
traffic_df = traffic_df[traffic_df['year_month'] == '2026-07']

traffic_df

,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour,year_month
288,admiralty mrt station,1297451.0,1148371.0,2445822.0,1948912.0,496910.0,545417.0,515983.0,7.0,2026-07
289,aljunied mrt station,725638.0,733965.0,1459603.0,1077763.0,381840.0,255792.0,296343.0,18.0,2026-07
290,ang mo kio mrt station,1274394.0,1408757.0,2683151.0,2104497.0,578654.0,499423.0,571260.0,18.0,2026-07
291,bartley mrt station,288510.0,255471.0,543981.0,443414.0,100567.0,124610.0,110367.0,7.0,2026-07
292,bayfront mrt station,1140191.0,1055521.0,2195712.0,1469332.0,726380.0,166977.0,354622.0,20.0,2026-07
...,...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,402335.0,373959.0,776294.0,637179.0,139115.0,176196.0,164702.0,7.0,2026-07
428,woodleigh mrt station,467808.0,480973.0,948781.0,741200.0,207581.0,167851.0,211816.0,18.0,2026-07
429,yew tee mrt station,919637.0,886547.0,1806184.0,1434612.0,371572.0,376052.0,383765.0,7.0,2026-07
430,yio chu kang mrt station,687769.0,640383.0,1328152.0,1157293.0,170859.0,346223.0,308066.0,8.0,2026-07


In [ ]:
scale = StandardScaler()
tdf = traffic_df.copy(deep = True)
tdf[num_col] = scale.fit_transform(traffic_df[num_col])
joblib.dump(scale, '/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/scalers/traffic_scale.pkl')
tdf = tdf.drop('year_month', axis = 1)
tdf

,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour
288,admiralty mrt station,1.397545,1.058514,1.227654,1.291102,0.943364,1.782587,1.098732,7.0
289,aljunied mrt station,0.229524,0.239906,0.235274,0.154663,0.472060,0.137620,0.115453,18.0
290,ang mo kio mrt station,1.350448,1.572875,1.466466,1.494068,1.278170,1.521358,1.346194,18.0
291,bartley mrt station,-0.663381,-0.705300,-0.686068,-0.672864,-0.679976,-0.607448,-0.717120,7.0
292,bayfront mrt station,1.076316,0.875100,0.975982,0.665476,1.883226,-0.366818,0.376355,20.0
...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,-0.430875,-0.471241,-0.452304,-0.420092,-0.522092,-0.314457,-0.473874,7.0
428,woodleigh mrt station,-0.297136,-0.259848,-0.278739,-0.284393,-0.241669,-0.361854,-0.262955,18.0
429,yew tee mrt station,0.625798,0.541313,0.584020,0.620183,0.430005,0.820654,0.506821,7.0
430,yio chu kang mrt station,0.152170,0.055046,0.103002,0.258412,-0.392075,0.651236,0.167934,8.0


In [199]:
tdf = traffic_df.copy(deep = True).drop('year_month', axis = 1)
tdf.columns = ['NAME', 'mrt_traffic_tap_in_amount_norm', 'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm', 'mrt_traffic_weekday_tap_norm',
       'mrt_traffic_weekend_tap_norm', 'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']

In [200]:
tdf.to_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/station-traffic-std.csv', index = False)

# Get Nearest Stations for each house

In [123]:
LAT0 = 1.35
R_EARTH = 6371000.0
def to_xy(lat, lon):
    x = np.radians(np.asarray(lon, float)) * math.cos(math.radians(LAT0)) * R_EARTH
    y = np.radians(np.asarray(lat, float)) * R_EARTH
    return np.column_stack([np.ravel(x), np.ravel(y)])

In [124]:
stations = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/station-keys.csv')[['NAME','latitude','longitude']]
houses = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/raw_graph_data/sg-hdb-block.csv')[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']]
houses

,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE
0,serangoon,106,serangoon north avenue 1,550106,1.370245,103.870823
1,bukit merah,34,telok blangah way,090034,1.273518,103.822481
2,punggol,226b,sumang lane,822226,1.402685,103.893609
3,marine parade,74,marine drive,440074,1.303705,103.907214
4,choa chu kang,284,choa chu kang avenue 3,NIL,1.379714,103.742138
...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,310155,1.332796,103.845589
9460,sembawang,307,canberra road,750307,1.445923,103.821657
9461,woodlands,681a,woodlands drive 62,731681,1.439534,103.803797
9462,choa chu kang,673a,choa chu kang crescent,681673,1.401014,103.747524


In [125]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(stations.latitude.values, stations.longitude.values)

In [126]:
# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

link_rows, count_rows = [], []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_station": stations.NAME[nearest],
        "nearest_station_m": round(float(d[nearest]), 1),
    }

    for WALK_M in [500,1000]:
        within = np.where(d <= WALK_M)[0]
        order = within[np.argsort(d[within])]  # nearest first
        for rank, m in enumerate(order, start=1):
            link_rows.append({
                "house_town": houses.TOWN[s], "house_block": houses.BLOCK[s], "house_street": houses.STREET[s], "station_name": stations.NAME[m],
                "distance_m": round(float(d[m]), 1), "rank": rank,
            })
        results[f"n_stations_{int(WALK_M)}m"] = len(order)
        # results[f"stations_{int(WALK_M)}m"]=  ";".join(stations.NAME[order].tolist())

    count_rows.append(results)

links = pd.DataFrame(link_rows).sort_values(["station_name", "rank"])
counts = pd.DataFrame(count_rows)

In [127]:
counts

,TOWN,BLOCK,STREET,nearest_station,nearest_station_m,n_stations_500m,n_stations_1000m
0,serangoon,106,serangoon north avenue 1,serangoon north mrt station,311.8,1,2
1,bukit merah,34,telok blangah way,harbourfront mrt station,916.7,0,1
2,punggol,226b,sumang lane,punggol mrt/lrt station,1029.6,0,0
3,marine parade,74,marine drive,marine parade mrt station,280.2,1,2
4,choa chu kang,284,choa chu kang avenue 3,choa chu kang west mrt station,313.2,1,2
...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,toa payoh mrt station,212.4,1,3
9460,sembawang,307,canberra road,sembawang mrt station,383.3,1,2
9461,woodlands,681a,woodlands drive 62,admiralty mrt station,343.3,1,1
9462,choa chu kang,673a,choa chu kang crescent,yew tee mrt station,390.7,1,2


# Join all features

In [272]:
# Keys, train and test set
train_house = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/train-house.csv')
test_house = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/test-house.csv')
stat_key = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/station-keys.csv')[[
    'NAME', 'nearest_mall', 'n_lines', 'status',
       'n_malls_500m', 'nearest_mall_m'
]]
stat_key.columns = ['nearest_station','nearest_mall','train_lines','status','n_malls_500m','nearest_mall_m']
stat_key['status'] = stat_key['status'].apply(lambda x: 1 if x == 'open' else 0)
stat_key.columns = ['nearest_station','nearest_mall','train_lines','station_is_open','n_malls_500m','nearest_mall_m']

# Train Traffic Features
traffic = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/station-traffic-std.csv')
traffic.columns = ['nearest_station', 'mrt_traffic_tap_in_amount_norm',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']

# Cluster Output
train_lap = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/train_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)
test_lap = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/test_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)

# Planning Area features
plan_area = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/data/collected_data/zoning_data/sg-demographics-planning-area.csv')[[
    'planning_area',
    'total_population', 
    'pct_age_0_14', 
    'pct_age_15_64', 
    'pct_age_65_plus',
    'unemployment_rate', 
    'total_households', 
    'pct_hdb',
    'pct_private_housing', 'pct_hh_income_ge_10k', 'pct_hh_income_below_2k',
    'avg_household_size', 'pct_chinese', 'pct_malay', 'pct_indian',
    'pct_owner_occupied', 'pct_pmet'
]]
plan_ft = []
for c in plan_area.columns:
    if c != 'planning_area':
        plan_ft.append(f'plan_area_ft_{c}')
    else:
        plan_ft.append('PLANNING_AREA')
plan_area.columns = plan_ft

# Subzone Area features
subzone = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/data/collected_data/zoning_data/sg-demographics-subzone.csv')[
    ['total_population', 'pct_age_0_14', 'pct_age_15_64', 'pct_age_65_plus', 'pct_male']
]

# Mall Features
mall = pd.read_csv('/Users/keltonn/Desktop/Repo/CS5228-project/data/collected_data/mall_features/sg-mall-features-wikipedia.csv')


In [273]:
# Join Cluster
print(f'Train: {train_house.shape}')
print(f'Test: {test_house.shape}')
train = pd.merge(train_house, train_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).reset_index(drop=True)
test = pd.merge(test_house, test_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).drop_duplicates().reset_index(drop=True)

# Join Planning Area
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, plan_area, how = 'left',on = ['PLANNING_AREA'])
test = pd.merge(test, plan_area, how = 'left',on = ['PLANNING_AREA'])

# Join nearest_station
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, counts, how = 'left',on = ['TOWN', 'BLOCK','STREET'])
test = pd.merge(test, counts, how = 'left',on = ['TOWN', 'BLOCK','STREET'])

# Join MRT Traffic
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, traffic, how = 'left',on = ['nearest_station'])
test = pd.merge(test, traffic, how = 'left',on = ['nearest_station'])

# Join Nearest Mall
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, stat_key, how = 'left',on = ['nearest_station'])
test = pd.merge(test, stat_key, how = 'left',on = ['nearest_station'])

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')

Train: (149359, 18)
Test: (42236, 17)
Train: (149359, 20)
Test: (42236, 19)
Train: (149359, 36)
Test: (42236, 35)
Train: (149359, 40)
Test: (42236, 39)
Train: (149359, 48)
Test: (42236, 47)
Train: (149359, 53)
Test: (42236, 52)


In [274]:
label = 'MONTHLY_RENT'

id_col = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'LEASE_COMMENCE_DATE',
       'TOWN', 'BLOCK', 'STREET',
       'POSTAL_CODE', 'LATITUDE', 'LONGITUDE', 'YEAR_COMPLETED',
       'SUBZONE', 'PLANNING_AREA', 'REGION', 'nearest_station', 'nearest_mall']

num_col = ['FLOOR_AREA_SQM','plan_area_ft_total_population',
       'plan_area_ft_pct_age_0_14', 'plan_area_ft_pct_age_15_64',
       'plan_area_ft_pct_age_65_plus', 'plan_area_ft_unemployment_rate',
       'plan_area_ft_total_households', 'plan_area_ft_pct_hdb',
       'plan_area_ft_pct_private_housing', 'plan_area_ft_pct_hh_income_ge_10k',
       'plan_area_ft_pct_hh_income_below_2k',
       'plan_area_ft_avg_household_size', 'plan_area_ft_pct_chinese',
       'plan_area_ft_pct_malay', 'plan_area_ft_pct_indian',
       'plan_area_ft_pct_owner_occupied', 'plan_area_ft_pct_pmet',
       'nearest_station_m', 'n_stations_500m',
       'n_stations_1000m', 'mrt_traffic_tap_in_amount_norm','n_malls_500m', 'nearest_mall_m',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm']

cat_col_num = ['MAX_FLOOR',
 'dbscan_cluster',
 'diffusion_cluster',
 'mrt_traffic_busiest_hour',
 'train_lines',
 'station_is_open']
cat_col = [i for i in test.columns if i not in num_col+id_col+[label]+cat_col_num]

In [275]:
# Numerical Columns
## Handle Null values
for n in num_col:
    train[n] = train[n].fillna(-1)
    test[n] = test[n].fillna(-1)    

## Standard Scaler
num_scale = StandardScaler()
train[num_col] = num_scale.fit_transform(train[num_col])
test[num_col] = num_scale.transform(test[num_col])
joblib.dump(num_scale, '/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/scalers/numerical_scale.pkl')


['/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/scalers/numerical_scale.pkl']

In [276]:
# Categorical Columns
for c in cat_col_num:
    train[c] = train[c].fillna(-1).apply(lambda x: int(x))
    test[c] = test[c].fillna(-1).apply(lambda x: int(x))

# One hot encode strings
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
enc.set_output(transform="pandas")

trt = enc.fit_transform(train[cat_col])
tst = enc.fit_transform(test[cat_col])
train = pd.concat([train,trt], axis = 1)
test = pd.concat([test,tst], axis = 1)
one_hot_col = ['FLAT_TYPE_1 room',
       'FLAT_TYPE_2 room', 'FLAT_TYPE_3 room', 'FLAT_TYPE_4 room',
       'FLAT_TYPE_5 room', 'FLAT_TYPE_executive', 'FLAT_MODEL_2 room',
       'FLAT_MODEL_3gen', 'FLAT_MODEL_adjoined flat', 'FLAT_MODEL_apartment',
       'FLAT_MODEL_dbss', 'FLAT_MODEL_improved',
       'FLAT_MODEL_improved maisonette', 'FLAT_MODEL_maisonette',
       'FLAT_MODEL_model a', 'FLAT_MODEL_model a maisonette',
       'FLAT_MODEL_model a2', 'FLAT_MODEL_new generation',
       'FLAT_MODEL_premium apartment', 'FLAT_MODEL_premium apartment loft',
       'FLAT_MODEL_premium maisonette', 'FLAT_MODEL_simplified',
       'FLAT_MODEL_standard', 'FLAT_MODEL_terrace', 'FLAT_MODEL_type s1',
       'FLAT_MODEL_type s2']

for c in one_hot_col:
    if c not in test.columns:
        test[c] = 0

In [277]:
final_train = train[id_col+num_col+cat_col_num+cat_col + one_hot_col + [label]]
final_test = test[id_col+num_col+cat_col_num+cat_col+one_hot_col]

In [278]:
final_train.to_csv('/Users/keltonn/Desktop/Repo/CS5228-project/99_predictions/data/train_feature.csv',index = False)
final_test.to_csv('/Users/keltonn/Desktop/Repo/CS5228-project/99_predictions/data/test_feature.csv',index = False)

In [280]:
final_train

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION,nearest_station,nearest_mall,FLOOR_AREA_SQM,plan_area_ft_total_population,plan_area_ft_pct_age_0_14,plan_area_ft_pct_age_15_64,plan_area_ft_pct_age_65_plus,plan_area_ft_unemployment_rate,plan_area_ft_total_households,plan_area_ft_pct_hdb,plan_area_ft_pct_private_housing,plan_area_ft_pct_hh_income_ge_10k,plan_area_ft_pct_hh_income_below_2k,plan_area_ft_avg_household_size,plan_area_ft_pct_chinese,plan_area_ft_pct_malay,plan_area_ft_pct_indian,plan_area_ft_pct_owner_occupied,plan_area_ft_pct_pmet,nearest_station_m,n_stations_500m,n_stations_1000m,mrt_traffic_tap_in_amount_norm,n_malls_500m,nearest_mall_m,mrt_traffic_tap_out_amount_norm,mrt_traffic_total_tap_norm,mrt_traffic_weekday_tap_norm,mrt_traffic_weekend_tap_norm,mrt_traffic_am_peak_norm,mrt_traffic_pm_peak_norm,MAX_FLOOR,dbscan_cluster,diffusion_cluster,mrt_traffic_busiest_hour,train_lines,station_is_open,FLAT_TYPE,FLAT_MODEL,FLAT_TYPE_1 room,FLAT_TYPE_2 room,FLAT_TYPE_3 room,FLAT_TYPE_4 room,FLAT_TYPE_5 room,FLAT_TYPE_executive,FLAT_MODEL_2 room,FLAT_MODEL_3gen,FLAT_MODEL_adjoined flat,FLAT_MODEL_apartment,FLAT_MODEL_dbss,FLAT_MODEL_improved,FLAT_MODEL_improved maisonette,FLAT_MODEL_maisonette,FLAT_MODEL_model a,FLAT_MODEL_model a maisonette,FLAT_MODEL_model a2,FLAT_MODEL_new generation,FLAT_MODEL_premium apartment,FLAT_MODEL_premium apartment loft,FLAT_MODEL_premium maisonette,FLAT_MODEL_simplified,FLAT_MODEL_standard,FLAT_MODEL_terrace,FLAT_MODEL_type s1,FLAT_MODEL_type s2,MONTHLY_RENT
0,2021,1,2012,punggol,604a,punggol road,821604,1.404310,103.907413,2010,waterway east,punggol,north-east region,punggol mrt/lrt station,Waterway Point,-0.016241,-0.047398,3.289609,-0.691610,-1.730771,-0.811468,-0.173978,0.481072,-0.476258,0.077117,-0.761529,0.517986,0.121965,0.288257,-1.181066,0.542282,0.588903,-0.104933,-0.844013,-0.581730,1.176743,0.477343,-0.828922,1.054924,1.117394,1.045435,1.353766,0.685497,1.022432,16,0,0,18,3,1,4 room,model a,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1800
1,2021,1,2001,sengkang,290c,compassvale crescent,543290,1.397018,103.897535,2000,sengkang town centre,sengkang,north-east region,sengkang mrt/lrt station,Compass One,0.684225,1.019913,1.819334,0.148653,-1.299997,-1.143692,0.886841,0.368027,-0.344994,0.077117,-0.876429,0.637434,0.388477,-0.295871,-0.156499,0.711681,0.270727,0.132757,-0.844013,-0.581730,1.225483,0.477343,-0.963672,1.240609,1.236464,1.242553,1.199175,1.102846,1.240234,15,0,0,18,2,1,5 room,improved,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000
2,2021,1,1976,queenstown,5,ghim moh road,270005,1.312693,103.788309,1975,ghim moh,queenstown,central region,buona vista mrt station,The Star Vista,-1.169949,-1.166069,-0.437442,-1.325959,1.131859,0.821966,-1.047497,0.244560,-0.223971,0.077117,0.617271,-0.843728,1.047745,-1.004211,-0.959376,-0.350109,0.886941,0.106755,-0.844013,0.252179,0.300439,0.477343,-0.776008,0.208634,0.254169,0.358550,-0.113052,0.484020,0.635972,15,1,1,18,2,1,3 room,improved,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000
3,2021,1,2010,sengkang,436b,fernvale road,792436,1.392878,103.876976,2008,fernvale,sengkang,north-east region,sengkang mrt/lrt station,Compass One,0.107371,1.019913,1.819334,0.148653,-1.299997,-1.143692,0.886841,0.368027,-0.344994,0.077117,-0.876429,0.637434,0.388477,-0.295871,-0.156499,0.711681,0.270727,4.523752,-0.844013,-1.415639,1.225483,0.477343,-0.963672,1.240609,1.236464,1.242553,1.199175,1.102846,1.240234,21,0,2,18,2,1,4 room,model a,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1400
4,2021,1,2001,sembawang,467b,admiralty drive,752467,1.451141,103.813630,1999,sembawang north,sembawang,north region,sembawang m